This notebook helps you experiment with Top-K retrieval and Similarity Thresholds using LlamaIndex and HuggingFace Embeddings.

# ✅ STEP 1: Setup

In [ ]:
!pip install llama-index pymupdf llama-index-embeddings-huggingface



INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.6/284.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.7/309.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.

In [ ]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.settings import Settings
import fitz  # PyMuPDF
import time

In [ ]:
# Load a sample PDF or plain text (You can upload your own contract PDF here)
pdf_path = "/content/sample_contract.pdf"
doc = fitz.open(pdf_path)
text = "\\n".join([page.get_text() for page in doc])
#documents = [Document(text=text)]

In [ ]:
# Configure the embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

In [ ]:
# Define a sentence splitter (can also use TokenTextSplitter or CharacterTextSplitter)
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

# Then create the index from these nodes
index = VectorStoreIndex(nodes)


# ✅ STEP 2: Experiment with Top-K Retrieval

In [ ]:
query = "What is the maximum loan amount a borrower can apply for?"
top_k_values = [2, 5, 10]

In [ ]:
for top_k in top_k_values:
    print(f"\\n--- Results for top_k = {top_k} ---")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"Result {i+1}:")
        print(node.get_text())
        print("-" * 80)

\n--- Results for top_k = 2 ---
Result 1:
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Result 2:
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier terminated as provided herein.
--------------------------------------------------------------------------------
\n--- Results for top_k = 5 ---
Result 1:
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Result 2:
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier termin

# ✅ STEP 3: Apply Similarity Thresholds

In [ ]:
# Retrieve nodes with top_k = 10
retriever = index.as_retriever(similarity_top_k=10)
retrieved_nodes = retriever.retrieve(query)

In [ ]:
# Try filtering by score
for threshold in [0.7, 0.75, 0.8]:
    filtered_nodes = [node for node in retrieved_nodes if node.score and node.score > threshold]
    print(f"\\n--- Results for threshold = {threshold} ---")
    print(f"Filtered {len(filtered_nodes)} out of {len(retrieved_nodes)} total nodes.")
    for i, node in enumerate(filtered_nodes):
        print(f"Chunk {i+1} (Score: {node.score:.2f}):")
        print(node.get_text())
        print("-" * 80)

\n--- Results for threshold = 0.7 ---
Filtered 0 out of 10 total nodes.
\n--- Results for threshold = 0.75 ---
Filtered 0 out of 10 total nodes.
\n--- Results for threshold = 0.8 ---
Filtered 0 out of 10 total nodes.


# ✅ STEP 4: Combined Configurations

In [ ]:
experiments = [
    {"top_k": 5, "threshold": None},
    {"top_k": 8, "threshold": 0.75},
    {"top_k": 5, "threshold": 0.8},
]



In [ ]:
for exp in experiments:
    print(f"\\n--- Experiment: top_k={exp['top_k']}, threshold={exp['threshold']} ---")
    retriever = index.as_retriever(similarity_top_k=exp["top_k"])
    nodes = retriever.retrieve(query)
    if exp["threshold"]:
        nodes = [node for node in nodes if node.score and node.score > exp["threshold"]]
    print(f"Chunks Retrieved: {len(nodes)}")
    for i, node in enumerate(nodes):
        print(f"Chunk {i+1} (Score: {node.score:.2f}):")
        print(node.get_text())
        print("-" * 80)

\n--- Experiment: top_k=5, threshold=None ---
Chunks Retrieved: 5
Chunk 1 (Score: 0.25):
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Chunk 2 (Score: 0.18):
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier terminated as provided herein.
--------------------------------------------------------------------------------
Chunk 3 (Score: 0.17):
3.2 Either party may terminate this Agreement upon thirty (30) days written notice to the other party.
4.
--------------------------------------------------------------------------------
Chunk 4 (Score: 0.14):
4.2 Refunds are issued at the sole discretion of Service Provider and will be processed within 30 days
of approval.
\n4.3 No refunds will be issued for co

✅ You can now adjust `top_k` and `threshold`, and try other queries!